<a href="https://colab.research.google.com/github/Netrahoni/FlyRankAi-Intern-work-Files/blob/main/work/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install DuckDB
!pip install duckdb -q

import duckdb
import pandas as pd
from google.colab import userdata

# 2. Get your secret token
hf_token = userdata.get('HF_TOKEN')

# 3. Connect to the database and authenticate with Hugging Face
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

# 4. Pull the data using the correct 2025/2026 dates
query = """
WITH historical_features AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as total_clicks,
        SUM(gsc_impressions) as total_impressions,
        AVG(gsc_avg_position) as avg_position,
        (SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0)) as ctr
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2025-11-01' AND '2026-04-30'
    GROUP BY content_hash_id
),
future_labels AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) as future_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-05-01' AND '2026-05-31'
    GROUP BY content_hash_id
)
SELECT
    h.*,
    f.future_clicks,
    CASE WHEN f.future_clicks < (h.total_clicks / 6.0) * 0.85 THEN 1 ELSE 0 END as needs_refresh
FROM historical_features h
JOIN future_labels f ON h.content_hash_id = f.content_hash_id
WHERE h.total_impressions > 1000
"""

# 5. Save it as a table called 'df' and print the results
df = con.execute(query).df()
print(f"Success! We downloaded {df.shape[0]} rows and {df.shape[1]} columns of data.")
df.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# 1. Fill any blank spaces in the data with 0 to prevent errors
df = df.fillna(0)

# 2. Define our Features (the clues) and our Label (the answer key)
features = ['total_clicks', 'total_impressions', 'avg_position', 'ctr']
X = df[features]
y = df['needs_refresh']

# 3. Split the data: 80% to train the model, 20% to test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Create a "Baseline" (a simple guess to see if our ML model is actually smart)
# Let's guess: "If a page's CTR is below average, it will decay and need a refresh"
average_ctr = X_train['ctr'].mean()
baseline_predictions = (X_test['ctr'] < average_ctr).astype(int)

# 5. Train the Random Forest Machine Learning Model
print("Training the Random Forest model... hang tight for a few seconds.\n")
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 6. Have the model predict on the 20% test data
model_predictions = model.predict(X_test)

# 7. Print the scorecards!
print("--- BASELINE HEURISTIC SCORECARD ---")
print(classification_report(y_test, baseline_predictions, zero_division=0))

print("\n--- RANDOM FOREST ML SCORECARD ---")
print(classification_report(y_test, model_predictions, zero_division=0))

In [ ]:
# 1. Get the model's confidence (from 0.0 to 1.0) that a page will decay
probabilities = model.predict_proba(X_test)[:, 1]

# 2. Make a copy of our test data so we can attach the scores to the page IDs
results = X_test.copy()
results['content_hash_id'] = df.loc[X_test.index, 'content_hash_id']
results['refresh_probability'] = probabilities

# 3. Calculate the Opportunity Score (Probability x Historical Impressions)
# This pushes high-traffic, high-risk pages to the very top!
results['opportunity_score'] = results['refresh_probability'] * results['total_impressions']

# 4. Sort the list from highest score to lowest
action_playbook = results.sort_values(by='opportunity_score', ascending=False)

# 5. Show the top 10 pages for the Editorial Team
print("--- TOP 10 PAGES FOR IMMEDIATE REFRESH ---")
action_playbook[['content_hash_id', 'total_impressions', 'refresh_probability', 'opportunity_score']].head(10)